# Natural Language Processing Lab - Assignment 02
## Task 3: WordNet Semantic Similarity & Shortest Path Metrics

**Objective**:
1. Select 8 word pairs exhibiting varying degrees of semantic relatedness.
2. Calculate shortest-path similarity scores (`path_similarity`) and Wu-Palmer similarity (`wup_similarity`).
3. Rank word pairs and analyze the relationship between semantic similarity and taxonomic distance.

In [ ]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import wordnet as wn

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Select benchmark word pairs
word_pairs = [
    ("car", "automobile"),
    ("gem", "jewel"),
    ("journey", "voyage"),
    ("dog", "cat"),
    ("dog", "animal"),
    ("bird", "airplane"),
    ("dog", "computer"),
    ("rooster", "voyage")
]

### 1. Computing Semantic Similarity Scores

In [ ]:
similarity_records = []

for w1, w2 in word_pairs:
    syn1 = wn.synsets(w1, pos=wn.NOUN)[0]
    syn2 = wn.synsets(w2, pos=wn.NOUN)[0]
    
    path_sim = syn1.path_similarity(syn2)
    wup_sim  = syn1.wup_similarity(syn2)
    lcs = syn1.lowest_common_hypernyms(syn2)
    lcs_name = lcs[0].name() if lcs else "None"
    
    similarity_records.append({
        "Pair": f"{w1} - {w2}",
        "Synset 1": syn1.name(),
        "Synset 2": syn2.name(),
        "Path Similarity": round(path_sim, 4) if path_sim else 0.0,
        "Wu-Palmer Sim": round(wup_sim, 4) if wup_sim else 0.0,
        "Lowest Common Subsumer": lcs_name
    })

sim_df = pd.DataFrame(similarity_records).sort_values(by="Path Similarity", ascending=False).reset_index(drop=True)
print("Semantic Similarity Rankings:")
display(sim_df)

### 2. Visualizing Semantic Similarity Rankings

In [ ]:
plt.figure(figsize=(10, 5), dpi=100)
sns.barplot(x="Path Similarity", y="Pair", data=sim_df, palette="crest")
plt.title("WordNet Path Similarity Across Benchmark Concept Pairs", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Path Similarity Score (0.0 to 1.0)", fontsize=11)
plt.ylabel("Word Pairs", fontsize=11)
plt.xlim(0, 1.05)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

### Observations:
1. **Synonyms (`car - automobile`, `gem - jewel`)**: Score identical or near-1.0 similarity because they map to the same synset or direct coordinates in the ontology.
2. **Co-hyponyms (`dog - cat`)**: High similarity as they share the immediate lowest common subsumer `carnivore.n.01`.
3. **Hypernym-Hyponym (`dog - animal`)**: High similarity reflecting direct vertical taxonomic ancestor link.
4. **Semantically Unrelated (`dog - computer`, `rooster - voyage`)**: Very low path similarity, as the shortest path between them must traverse all the way up through the root `entity.n.01` node.